# Data Preprocessing PipelineThis notebook downloads the dataset from Kaggle (if not already present) and prepares it for the model notebooks.

In [ ]:
import osimport subprocessimport shutilimport reimport pandas as pdimport numpy as npfrom pathlib import Pathfrom tqdm.auto import tqdmimport yaml# Load configwith open("config.yaml", 'r') as f:    config = yaml.safe_load(f)

# 1. Download Dataset from Kaggle (if not exists)

In [ ]:
# Paths from configDATASET_DIR = Path(config['paths']['dataset_dir'])IMAGES_DIR = DATASET_DIR / config['paths']['images_root']METADATA_FILE = DATASET_DIR / config['paths']['metadata_path']# Raw data paths (downloaded from Kaggle)RAW_DATA_DIR = Path("data/raw/mlops-torob-dataset")IMAGE_SOURCE_DIR = Path("data/raw/torob-images/balanced_images")def download_kaggle_dataset(slug, dest_dir):    """Download a Kaggle dataset if it doesn't exist locally."""    if dest_dir.exists() and any(dest_dir.iterdir()):        print(f"   ✅ Dataset already exists: {dest_dir}")        return    dest_dir.mkdir(parents=True, exist_ok=True)    print(f"   ⬇️  Downloading {slug}...")    subprocess.run(        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest_dir), "--unzip"],        check=True    )    print(f"   ✅ Downloaded to {dest_dir}")# Download datasets if not presentprint("📦 Checking datasets...")# Check if final processed data already existsif METADATA_FILE.exists() and IMAGES_DIR.exists():    print("✅ Processed dataset already exists. Skipping download & preprocessing.")    print(f"   - Metadata: {METADATA_FILE}")    print(f"   - Images: {IMAGES_DIR}")else:    download_kaggle_dataset(config['dataset']['kaggle_dataset_slug'], RAW_DATA_DIR)    download_kaggle_dataset(config['dataset']['kaggle_images_slug'], IMAGE_SOURCE_DIR.parent)

# 2. Text Normalization

In [ ]:
def clean_persian_text(text):    """    Converts Persian/Arabic numbers to English, removes diacritics,     removes punctuation, and standardizes characters.    """    if not isinstance(text, str): return ""        # 1. Map Persian/Arabic Digits to English    persian_digits = "\u06F0\u06F1\u06F2\u06F3\u06F4\u06F5\u06F6\u06F7\u06F8\u06F9"    arabic_digits = "\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669"    english_digits = "0123456789"        trans_table = str.maketrans(persian_digits + arabic_digits, english_digits * 2)    text = text.translate(trans_table)        # 2. Fix Number Formatting (Remove commas inside numbers: 12,000 -> 12000)    text = re.sub(r'(\d),(\d{3})', r'\1\2', text)        # 3. Remove Diacritics (Fatha, Kasra, etc.)    text = re.sub(r'[\u064B-\u0652]', '', text)        # 4. Standardize Characters (Ye/Ke/ZWNJ)    text = text.replace('\u064A', '\u06CC').replace('\u0643', '\u06A9').replace('\u0649', '\u06CC')    text = text.replace('\u200c', ' ')  # ZWNJ -> Space        # 5. Remove Special Symbols (Keep only word chars, spaces)    text = re.sub(r'[^\w\s]', ' ', text)        # 6. Collapse Whitespace    return re.sub(r'\s+', ' ', text).strip()# Testsample_text = "\u0633\u0644\u0627\u0645! \u0645\u0646 \u06A9\u0650\u062A\u0627\u0628\u0650 \u00AB\u062C\u0646\u06AF \u0648 \u0635\u0644\u062D\u00BB \u0631\u0627 \u062F\u0648\u0633\u062A \u062F\u0627\u0631\u0645. (\u0642\u06CC\u0645\u062A: \u06F1\u06F2,\u06F0\u06F0\u06F0 \u062A\u0648\u0645\u0627\u0646)"print(f"\n\U0001F9EA Normalization Test:")print(f"   Original: {sample_text}")print(f"   Cleaned:  {clean_persian_text(sample_text)}")

# 3. Synchronize Images & Metadata

In [ ]:
# Skip if already processedif METADATA_FILE.exists() and IMAGES_DIR.exists():    print("\u2705 Already processed. Loading existing data...")    final_dataset = pd.read_parquet(METADATA_FILE)    print(f"   - {len(final_dataset):,} products ready.")else:    print("\n\U0001F504 Synchronizing Data...")        # A. Scan Images    image_files = list(IMAGE_SOURCE_DIR.glob("*.jpg"))    available_keys = {f.stem for f in image_files}    print(f"   - Found {len(available_keys):,} images.")        # B. Load Metadata    print("   - Loading Parquet files...")    df_products = pd.read_parquet(RAW_DATA_DIR / "base_products.parquet")    df_cats = pd.read_parquet(RAW_DATA_DIR / "categories.parquet")    df_members = pd.read_parquet(RAW_DATA_DIR / "members.parquet")        # C. Calculate Mean Price    price_stats = df_members.groupby('base_random_key')['price'].mean().reset_index()        # D. Merge & Filter    full_df = df_products.merge(price_stats, left_on='random_key', right_on='base_random_key', how='left')    full_df = full_df.merge(df_cats[['id', 'title']], left_on='category_id', right_on='id', how='left')    full_df.rename(columns={'title': 'category_name', 'mean': 'price'}, inplace=True)        filtered_df = full_df[full_df['random_key'].isin(available_keys)].copy()    print(f"   - Matched {len(filtered_df):,} rows.")        # E. Apply Cleaning    print("\n\U0001F9FD Applying Normalization...")    filtered_df['clean_name'] = filtered_df['persian_name'].apply(clean_persian_text)    filtered_df['clean_category'] = filtered_df['category_name'].fillna("").apply(clean_persian_text)    filtered_df['unified_text'] = filtered_df['clean_name'] + " " + filtered_df['clean_category']    filtered_df['image_path'] = filtered_df['random_key'].apply(lambda x: f"images/{x}.jpg")        # F. Select final schema    final_cols = ['random_key', 'image_path', 'clean_name', 'unified_text', 'price', 'category_name']    final_dataset = filtered_df[final_cols].copy()        # G. Save    DATASET_DIR.mkdir(parents=True, exist_ok=True)    IMAGES_DIR.mkdir(parents=True, exist_ok=True)    final_dataset.to_parquet(METADATA_FILE, index=False)    print(f"   \u2705 Metadata saved: {METADATA_FILE}")        # H. Copy Images    print("\n\U0001F4E6 Copying Images...")    for img_path in tqdm(image_files, desc="Copying Images"):        dest_path = IMAGES_DIR / img_path.name        if not dest_path.exists():            shutil.copy2(img_path, dest_path)        print(f"\n\U0001F389 Dataset Ready: {DATASET_DIR}")    print(f"   \u251C\u2500\u2500 processed_metadata.parquet  ({len(final_dataset)} rows)")    print(f"   \u2514\u2500\u2500 images/                     ({len(image_files)} images)")